# LeetCode 939 — Minimum Area Rectangle bằng Python

Link đề: https://leetcode.com/problems/minimum-area-rectangle/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Math, Geometry, Sorting

Skeleton LeetCode:

```python
class Solution:
    def minAreaRect(self, points: List[List[int]]) -> int:
        ...
```

Ghi chú: đáp án duy nhất (một số nguyên; `0` nếu không có hình chữ nhật). Điểm cốt lõi: hình chữ nhật **song song các trục** được xác định bởi **hai đường chéo** `(x1, y1)`–`(x2, y2)` với `x1 ≠ x2`, `y1 ≠ y2`; hai đỉnh còn lại `(x1, y2)`, `(x2, y1)` phải có trong tập điểm. Với `n ≤ 500` duyệt mọi cặp điểm làm đường chéo (`O(n²)`) và tra tập băm là đủ; hoặc **nhóm theo `x`** rồi nhớ "cặp `(y1, y2)` xuất hiện lần cuối ở cột nào". Oracle: **vét cạn mọi bộ 4 điểm** kiểm tra có tạo thành hình chữ nhật song song trục.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho các điểm **phân biệt** trên mặt phẳng, tìm diện tích **nhỏ nhất** của hình chữ nhật có cạnh song song các trục mà **bốn đỉnh đều thuộc tập điểm**.
2. Nắm công cụ Python: `set` các điểm dạng `tuple`, nhóm điểm theo `x` bằng `defaultdict`, `itertools.combinations`, khoá là **cặp** `(y1, y2)`, `float('inf')`.
3. Đi từ **vét cạn bộ 4 điểm** → **đường chéo + tra tập băm (`O(n²)`)** → **nhóm theo cột, nhớ cặp `(y1, y2)`** → **duyệt cặp cột và giao tập `y`**.
4. Hiểu mấu chốt: hình chữ nhật song song trục ⇔ hai cột `x1 < x2` cùng chứa hai cao độ `y1 < y2`; diện tích `(x2 − x1)(y2 − y1)`.
5. Tránh lỗi kinh điển: nhận cả cặp có `x1 == x2` hoặc `y1 == y2` (diện tích `0`, không phải hình chữ nhật), trả `inf` thay vì `0` khi không có, lưu **cột đầu tiên** thay vì **gần nhất** của cặp `(y1, y2)` (bỏ lỡ hình chữ nhật nhỏ nhất), chỉ xét các cặp `y` kề nhau, và tính diện tích sai dấu.

## 2. Đề bài & ví dụ

Cho mảng `points` các điểm trên mặt phẳng `X–Y`, `points[i] = [xi, yi]`.

Trả về **diện tích nhỏ nhất** của một hình chữ nhật tạo từ các điểm này với các cạnh **song song** các trục `X` và `Y`. Nếu không có hình chữ nhật nào, trả về `0`.

**Ví dụ 1:** `points = [[1,1],[1,3],[3,1],[3,3],[2,2]]` → `4` (hình vuông có đỉnh `(1,1),(1,3),(3,1),(3,3)`).

**Ví dụ 2:** `points = [[1,1],[1,3],[3,1],[3,3],[4,1],[4,3]]` → `2` (hình chữ nhật `x ∈ [3, 4]`, `y ∈ [1, 3]`).

**Ràng buộc:** `1 <= points.length <= 500`; `0 <= xi, yi <= 4·10^4`; mọi điểm **phân biệt**.

In [1]:
import itertools
import random
import time
from collections import defaultdict
from typing import List

examples = [([[1, 1], [1, 3], [3, 1], [3, 3], [2, 2]], 4), ([[1, 1], [1, 3], [3, 1], [3, 3], [4, 1], [4, 3]], 2)]
for pts, want in examples:
    print(pts, "->", want)

[[1, 1], [1, 3], [3, 1], [3, 3], [2, 2]] -> 4
[[1, 1], [1, 3], [3, 1], [3, 3], [4, 1], [4, 3]] -> 2


## 3. Kiến thức nền

### 3.1 Hình chữ nhật song song trục

Hình chữ nhật song song trục có bốn đỉnh `(xa, ya), (xa, yb), (xb, ya), (xb, yb)` với `xa < xb`, `ya < yb`: hai đỉnh **chéo nhau** `(xa, ya)` và `(xb, yb)` xác định đủ hai đỉnh còn lại `(xa, yb)` và `(xb, ya)`. Diện tích `(xb − xa) · (yb − ya)` (luôn `> 0` vì `xa ≠ xb`, `ya ≠ yb`).

In [2]:
xa, ya, xb, yb = 1, 1, 3, 3
corners = {(xa, ya), (xa, yb), (xb, ya), (xb, yb)}
print(sorted(corners), "diện tích =", (xb - xa) * (yb - ya))
print("hai đường chéo xác định đủ bốn đỉnh:", {(xa, yb), (xb, ya)} == corners - {(xa, ya), (xb, yb)})

[(1, 1), (1, 3), (3, 1), (3, 3)] diện tích = 4
hai đường chéo xác định đủ bốn đỉnh: True


### 3.2 Tập băm các điểm

`S = {(x, y) for x, y in points}` cho phép kiểm tra "điểm có tồn tại không" trong `O(1)`. Điểm là `tuple` (băm được); `list` thì không.

### 3.3 Nhóm theo cột

`cols[x]` = danh sách (đã sắp xếp) các `y` của các điểm có hoành độ `x`. Hai điểm cùng cột là hai cạnh dọc của hình chữ nhật; cột khác chứa **cùng** hai cao độ tạo cạnh dọc đối diện.

### 3.4 Cỡ bài toán

`n ≤ 500`: `n²/2 = 125 000` cặp (rất nhẹ); `n⁴/24 ≈ 2.6·10^9` bộ 4 điểm (không thể). Vì vậy cần ít nhất `O(n²)`.

In [3]:
pts = examples[0][0]
S = {tuple(p) for p in pts}
cols = defaultdict(list)
for x, y in pts:
    cols[x].append(y)
print("tập:", sorted(S), "| cột:", {x: sorted(ys) for x, ys in cols.items()})
from math import comb
print("C(500,2) =", comb(500, 2), "| C(500,4) =", comb(500, 4))

tập: [(1, 1), (1, 3), (2, 2), (3, 1), (3, 3)] | cột: {1: [1, 3], 3: [1, 3], 2: [2]}
C(500,2) = 124750 | C(500,4) = 2573031125


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: các điểm `(1,1), (1,3), (3,1), (3,3), (2,2)`. Lấy đường chéo `(1,1)–(3,3)` (`x1 < x2`, `y1 < y2`): hai đỉnh còn lại `(1,3)` và `(3,1)` đều có mặt ⇒ hình chữ nhật diện tích `2 · 2 = 4`. Điểm `(2,2)` không tạo hình chữ nhật nào.

Ví dụ 2: thêm cột `x = 4` với `y = 1, 3`. Đường chéo `(3,1)–(4,3)` có đỉnh còn lại `(3,3)` và `(4,1)` ✓ ⇒ diện tích `1 · 2 = 2` — nhỏ hơn `4`.

Ta cần xét **mọi** đường chéo hợp lệ và lấy diện tích nhỏ nhất.

In [4]:
pts = examples[1][0]
S = {tuple(p) for p in pts}
best = None
for (x1, y1), (x2, y2) in itertools.combinations(sorted(S), 2):
    if x1 < x2 and y1 < y2 and (x1, y2) in S and (x2, y1) in S:
        area = (x2 - x1) * (y2 - y1)
        print(f"đường chéo ({x1},{y1})-({x2},{y2}): đỉnh còn lại có đủ, diện tích {area}")
        best = area if best is None else min(best, area)
print("nhỏ nhất:", best)

đường chéo (1,1)-(3,3): đỉnh còn lại có đủ, diện tích 4
đường chéo (1,1)-(4,3): đỉnh còn lại có đủ, diện tích 6
đường chéo (3,1)-(4,3): đỉnh còn lại có đủ, diện tích 2
nhỏ nhất: 2


## 5. Cách 1 — Brute Force: mọi bộ 4 điểm

**Ý tưởng:** với mỗi bộ 4 điểm phân biệt, kiểm tra có tạo thành hình chữ nhật song song trục: đúng `2` giá trị `x` khác nhau, đúng `2` giá trị `y` khác nhau, và 4 điểm là mọi tổ hợp. Lấy diện tích nhỏ nhất.

In [5]:
def min_area_brute(points):
    best = float("inf")
    for quad in itertools.combinations(points, 4):
        xs = sorted({p[0] for p in quad})
        ys = sorted({p[1] for p in quad})
        if len(xs) == 2 and len(ys) == 2 and len({tuple(p) for p in quad}) == 4:
            best = min(best, (xs[1] - xs[0]) * (ys[1] - ys[0]))      # 4 điểm phân biệt trên lưới 2x2 => đủ 4 góc
    return 0 if best == float("inf") else best


for pts, want in examples:
    assert min_area_brute(pts) == want
print([min_area_brute(p) for p, _ in examples])

[4, 2]


In [6]:
# Code nộp LeetCode cho Cách 1 (O(n^4): TLE với n = 500)
class SolutionBrute:
    def minAreaRect(self, points: List[List[int]]) -> int:
        best = float("inf")
        for quad in itertools.combinations(points, 4):
            xs = sorted({p[0] for p in quad})
            ys = sorted({p[1] for p in quad})
            if len(xs) == 2 and len(ys) == 2:
                best = min(best, (xs[1] - xs[0]) * (ys[1] - ys[0]))
        return 0 if best == float("inf") else best

**Độ phức tạp:** `O(n⁴)`. (Bốn điểm phân biệt có đúng hai giá trị `x` và hai giá trị `y` ⇒ chúng phủ 4 góc của lưới `2 × 2`.)

**Nút thắt:** hai điểm chéo đã xác định hai điểm còn lại; chỉ cần **tra** chúng trong tập băm.

## 6. Ý tưởng thuật toán / bước đột phá

Duyệt mọi cặp `(p, q)` làm đường chéo với `p.x < q.x`, `p.y < q.y`; nếu `(p.x, q.y)` và `(q.x, p.y)` cùng thuộc tập thì cập nhật đáp án. `O(n²)` cặp × `O(1)` tra cứu.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** đường chéo + tập băm, **Cách 3** nhóm theo cột và nhớ cặp `(y1, y2)`, **Cách 4** duyệt cặp cột và giao tập `y`.

### Cách 2 — Đường chéo + tập băm

**Giải thích từng đoạn:**

1. `S = set(map(tuple, points))`.
2. Với mọi cặp điểm `(x1, y1), (x2, y2)`: nếu `x1 < x2` và `y1 < y2` và `(x1, y2) ∈ S` và `(x2, y1) ∈ S` thì ứng viên `(x2 − x1)(y2 − y1)`.
3. Trả `0` nếu không có ứng viên.

In [7]:
def min_area_diagonal(points):
    S = {tuple(p) for p in points}
    best = float("inf")
    for (x1, y1), (x2, y2) in itertools.combinations(sorted(S), 2):      # sorted: luôn có x1 <= x2
        if x1 < x2 and y1 < y2 and (x1, y2) in S and (x2, y1) in S:       # đường chéo hợp lệ và hai đỉnh còn lại có mặt
            best = min(best, (x2 - x1) * (y2 - y1))
    return 0 if best == float("inf") else best


for pts, want in examples:
    assert min_area_diagonal(pts) == want
print([min_area_diagonal(p) for p, _ in examples])

[4, 2]


In [8]:
# Code nộp LeetCode cho Cách 2
class SolutionDiagonal:
    def minAreaRect(self, points: List[List[int]]) -> int:
        S = {tuple(p) for p in points}
        best = float("inf")
        for i, (x1, y1) in enumerate(points):
            for x2, y2 in points[i + 1:]:
                if x1 != x2 and y1 != y2 and (x1, y2) in S and (x2, y1) in S:
                    best = min(best, abs(x2 - x1) * abs(y2 - y1))
        return 0 if best == float("inf") else best

**Độ phức tạp:** `O(n²)` thời gian, `O(n)` bộ nhớ. Trong bản nộp dùng `x1 != x2 and y1 != y2` và `abs` thay cho sắp xếp (mỗi hình chữ nhật được xét qua cả hai đường chéo, không ảnh hưởng đáp án).

### Cách 3 — Nhóm theo cột, nhớ cặp `(y1, y2)`

**Ý tưởng:** xử lý các cột theo `x` tăng dần. Với mỗi cột, lấy các `y` đã sắp xếp, xét mọi **cặp** `(y1, y2)` trong cột: nếu cặp này đã xuất hiện ở cột `x_prev` (gần nhất trước đó) thì có hình chữ nhật diện tích `(x − x_prev)·(y2 − y1)`; rồi **cập nhật** `last[(y1, y2)] = x`. Lưu cột **gần nhất** là đủ vì cột xa hơn cho diện tích lớn hơn.

In [9]:
def min_area_columns(points):
    cols = defaultdict(list)
    for x, y in points:
        cols[x].append(y)
    last = {}                                            # (y1, y2) -> cột gần nhất đã có cả hai điểm
    best = float("inf")
    for x in sorted(cols):
        ys = sorted(cols[x])
        for i in range(len(ys)):
            for j in range(i + 1, len(ys)):
                pair = (ys[i], ys[j])
                if pair in last:
                    best = min(best, (x - last[pair]) * (ys[j] - ys[i]))
                last[pair] = x                           # cột gần nhất (ghi đè): diện tích nhỏ hơn cột cũ hơn
    return 0 if best == float("inf") else best


for pts, want in examples:
    assert min_area_columns(pts) == want
print([min_area_columns(p) for p, _ in examples])

[4, 2]


In [10]:
# Code nộp LeetCode cho Cách 3
class SolutionColumns:
    def minAreaRect(self, points: List[List[int]]) -> int:
        cols = defaultdict(list)
        for x, y in points:
            cols[x].append(y)
        last, best = {}, float("inf")
        for x in sorted(cols):
            ys = sorted(cols[x])
            for i in range(len(ys)):
                for j in range(i + 1, len(ys)):
                    pair = (ys[i], ys[j])
                    if pair in last:
                        best = min(best, (x - last[pair]) * (ys[j] - ys[i]))
                    last[pair] = x
        return 0 if best == float("inf") else best

**Độ phức tạp:** `Σ_x k_x²/2` với `k_x` số điểm trong cột `x`; tối đa `O(n²)` (khi mọi điểm thẳng hàng một cột) nhưng không bao giờ tệ hơn Cách 2; bộ nhớ `O(n²)` trong trường hợp xấu (số cặp `(y1, y2)` lưu).

### Cách 4 — Duyệt cặp cột và giao tập `y`

**Ý tưởng:** với mọi cặp cột `x1 < x2`, lấy giao của tập `y` hai cột, sắp xếp: hình chữ nhật nhỏ nhất giữa hai cột này dùng hai `y` **liền nhau** trong giao (khoảng cách nhỏ nhất). Diện tích `(x2 − x1) · min_gap`.

In [11]:
def min_area_column_pairs(points):
    cols = defaultdict(set)
    for x, y in points:
        cols[x].add(y)
    xs = sorted(cols)
    best = float("inf")
    for a in range(len(xs)):
        for b in range(a + 1, len(xs)):
            common = sorted(cols[xs[a]] & cols[xs[b]])
            if len(common) >= 2:
                gap = min(common[k + 1] - common[k] for k in range(len(common) - 1))     # hai y kề nhau trong giao
                best = min(best, (xs[b] - xs[a]) * gap)
    return 0 if best == float("inf") else best


for pts, want in examples:
    assert min_area_column_pairs(pts) == want
print([min_area_column_pairs(p) for p, _ in examples])

[4, 2]


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionColumnPairs:
    def minAreaRect(self, points: List[List[int]]) -> int:
        cols = defaultdict(set)
        for x, y in points:
            cols[x].add(y)
        xs = sorted(cols)
        best = float("inf")
        for a in range(len(xs)):
            for b in range(a + 1, len(xs)):
                common = sorted(cols[xs[a]] & cols[xs[b]])
                if len(common) >= 2:
                    gap = min(common[k + 1] - common[k] for k in range(len(common) - 1))
                    best = min(best, (xs[b] - xs[a]) * gap)
        return 0 if best == float("inf") else best

**Độ phức tạp:** `O(C² · k)` với `C` số cột khác nhau; tốt khi ít cột (nếu nhiều cột hơn hàng, hãy hoán vị trục để số cặp nhỏ hơn). Tối đa `O(n²)` vì `Σ|cột| = n`.

## 8. TRACE từng bước

**Trace Cách 3** với ví dụ 2 (`points = [[1,1],[1,3],[3,1],[3,3],[4,1],[4,3]]`):

| cột `x` | `ys` | cặp `(y1,y2)` | `last` trước | diện tích | `last` sau |
|---|---|---|---|---|---|
| 1 | `[1, 3]` | `(1,3)` | `{}` | — | `{(1,3): 1}` |
| 3 | `[1, 3]` | `(1,3)` | `{(1,3): 1}` | `(3 − 1)·2 = 4` | `{(1,3): 3}` |
| 4 | `[1, 3]` | `(1,3)` | `{(1,3): 3}` | `(4 − 3)·2 = 2` | `{(1,3): 4}` |

`best = min(4, 2) = 2` ✓. Hàm trace:

In [13]:
def trace_columns(points):
    cols = defaultdict(list)
    for x, y in points:
        cols[x].append(y)
    last, best = {}, float("inf")
    print(f"{'x':>3} {'ys':<12} {'cặp':<8} {'last[cặp] trước':>16} {'diện tích':>10}")
    for x in sorted(cols):
        ys = sorted(cols[x])
        for i in range(len(ys)):
            for j in range(i + 1, len(ys)):
                pair = (ys[i], ys[j])
                area = (x - last[pair]) * (ys[j] - ys[i]) if pair in last else None
                print(f"{x:>3} {str(ys):<12} {str(pair):<8} {str(last.get(pair)):>16} {str(area):>10}")
                if area is not None:
                    best = min(best, area)
                last[pair] = x
    print("=> nhỏ nhất:", 0 if best == float("inf") else best)
    return 0 if best == float("inf") else best


assert trace_columns(examples[1][0]) == 2

  x ys           cặp       last[cặp] trước  diện tích
  1 [1, 3]       (1, 3)               None       None
  3 [1, 3]       (1, 3)                  1          4
  4 [1, 3]       (1, 3)                  3          2
=> nhỏ nhất: 2


## 9. Code nộp LeetCode

Chọn **Cách 2** (đường chéo + tập băm, `O(n²)`, ngắn gọn). Giữ nguyên `class Solution`, `minAreaRect(self, points)`:

In [14]:
class Solution:
    def minAreaRect(self, points: List[List[int]]) -> int:
        S = {(x, y) for x, y in points}                     # tập điểm để tra cứu O(1)
        best = float("inf")
        n = len(points)
        for i in range(n):
            x1, y1 = points[i]
            for j in range(i + 1, n):
                x2, y2 = points[j]
                # (x1,y1) và (x2,y2) là hai đỉnh chéo nhau; hai đỉnh còn lại là (x1,y2) và (x2,y1)
                if x1 != x2 and y1 != y2 and (x1, y2) in S and (x2, y1) in S:
                    best = min(best, abs(x2 - x1) * abs(y2 - y1))
        return 0 if best == float("inf") else best


for pts, want in examples:
    assert Solution().minAreaRect([list(p) for p in pts]) == want
print([Solution().minAreaRect([list(p) for p in pts]) for pts, _ in examples])

[4, 2]


## 10. Vì sao đúng

**(a) Hình chữ nhật ↔ đường chéo.** Mọi hình chữ nhật song song trục có bốn đỉnh `(xa, ya), (xa, yb), (xb, ya), (xb, yb)` với `xa < xb`, `ya < yb`. Cặp `{(xa, ya), (xb, yb)}` (và cặp `{(xa, yb), (xb, ya)}`) xác định hình đó. Ngược lại, hai điểm `p = (x1, y1)`, `q = (x2, y2)` với `x1 ≠ x2`, `y1 ≠ y2` là hai đỉnh chéo nhau của đúng một hình chữ nhật song song trục, có hai đỉnh còn lại `(x1, y2)`, `(x2, y1)`. Hình chữ nhật có **cả bốn đỉnh trong tập** ⇔ tồn tại đường chéo `(p, q)` với `(x1, y2), (x2, y1) ∈ S`. Duyệt mọi cặp và tra tập tìm ra mọi hình chữ nhật (mỗi hình xuất hiện hai lần, ứng với hai đường chéo).

**(b) Diện tích dương.** Điều kiện `x1 ≠ x2` và `y1 ≠ y2` loại cặp "thoái hoá" (cùng hàng/cột) cho diện tích `0`, không phải hình chữ nhật thật.

**(c) Cách 3 đủ.** Với một cặp cao độ `(y1, y2)` và các cột `c_1 < c_2 < … < c_m` chứa cả hai, các hình chữ nhật dùng đúng cặp `y` này có diện tích `(c_b − c_a)·(y2 − y1)` với `a < b`; nhỏ nhất khi `b = a + 1`. Thuật toán so sánh cột hiện tại với cột **gần nhất trước nó** chứa cặp (nhờ ghi đè `last`), nên mọi cặp `(c_a, c_{a+1})` kề nhau đều được xét. Các cặp cột không kề cho diện tích lớn hơn nên bỏ qua không làm mất đáp án.

**(d) Cách 4.** Với hai cột `x1 < x2`, mọi hình chữ nhật có hai cạnh dọc trên hai cột này chọn hai `y` trong giao `Y₁ ∩ Y₂`; diện tích `(x2 − x1)·(y_b − y_a)` nhỏ nhất khi `y_a, y_b` là hai phần tử **kề nhau** trong giao đã sắp xếp (khoảng cách nhỏ nhất).

**(e) Cách 1** là định nghĩa trực tiếp: bốn điểm phân biệt với đúng hai giá trị `x` và hai giá trị `y` là bốn góc của lưới `2 × 2`.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **mọi bộ 4 điểm** trên mọi tập con của lưới nhỏ (mục 12D) và ngẫu nhiên (12E); và khớp nhau ở `n = 500` (12F).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi bộ 4 điểm | `O(n⁴)` | `O(1)` | oracle |
| 2. Đường chéo + tập băm | `O(n²)` | `O(n)` | **đáp án chính** |
| 3. Nhóm theo cột + `last` | `O(Σ k_x²)` | `O(n²)` xấu nhất | tốt khi nhiều cột nhỏ |
| 4. Cặp cột + giao tập `y` | `O(C²·k)` | `O(n)` | tốt khi ít cột |

Đo thực tế (`n = 500`; cách 1 chỉ `n = 24`):

In [15]:
def bench(fn, arg, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(arg)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(939)


def random_points(rng, n, X, Y):
    pts = set()
    while len(pts) < n:
        pts.add((rng.randint(0, X), rng.randint(0, Y)))
    return [list(p) for p in pts]


for label, (X, Y) in [("lưới thưa 40000²", (40000, 40000)), ("lưới đặc 25²", (24, 24)), ("dải 5 x 200", (4, 199))]:
    pts = random_points(rng, 500, X, Y)
    row = [f"{name} {bench(fn, pts) * 1e3:8.2f} ms" for name, fn in
           [("diagonal", min_area_diagonal), ("columns", min_area_columns), ("colpairs", min_area_column_pairs)]]
    print(f"{label:18s}: " + " | ".join(row))
print(f"cách 1 với n = 24: {bench(min_area_brute, random_points(rng, 24, 8, 8), 1) * 1e3:8.1f} ms (O(n^4))")

lưới thưa 40000²  : diagonal    13.01 ms | columns     0.44 ms | colpairs    51.52 ms
lưới đặc 25²      : diagonal    20.16 ms | columns     1.81 ms | colpairs     1.30 ms
dải 5 x 200       : diagonal    14.89 ms | columns     7.59 ms | colpairs     0.18 ms
cách 1 với n = 24:     15.0 ms (O(n^4))


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi tập con của lưới nhỏ với oracle bộ 4 điểm, (E) ngẫu nhiên có seed + tính chất, (F) giới hạn (`n = 500`).

### A. Ví dụ của đề

In [16]:
def oracle(points):
    """Đúng định nghĩa: thử mọi bộ 4 điểm; hợp lệ nếu có đúng 2 giá trị x, đúng 2 giá trị y và đủ 4 góc (độc lập với việc tra tập)."""
    best = None
    for quad in itertools.combinations([tuple(p) for p in points], 4):
        xs, ys = sorted({p[0] for p in quad}), sorted({p[1] for p in quad})
        if len(xs) == 2 and len(ys) == 2 and {(x, y) for x in xs for y in ys} == set(quad):
            area = (xs[1] - xs[0]) * (ys[1] - ys[0])
            best = area if best is None else min(best, area)
    return best or 0


IMPLS = {
    "brute": min_area_brute,
    "diagonal": min_area_diagonal,
    "columns": min_area_columns,
    "colpairs": min_area_column_pairs,
    "Solution": lambda p: Solution().minAreaRect([list(q) for q in p]),
    "SolutionBrute": lambda p: SolutionBrute().minAreaRect([list(q) for q in p]),
    "SolutionDiagonal": lambda p: SolutionDiagonal().minAreaRect([list(q) for q in p]),
    "SolutionColumns": lambda p: SolutionColumns().minAreaRect([list(q) for q in p]),
    "SolutionColumnPairs": lambda p: SolutionColumnPairs().minAreaRect([list(q) for q in p]),
}


def check(points, want=None, names=None):
    if want is None:
        want = oracle(points)
    for name in (names or IMPLS):
        got = IMPLS[name](points)
        assert got == want, (name, points, got, want)


for pts, want in examples:
    assert oracle(pts) == want
    check(pts, want)
print("ví dụ đề: oracle bộ-4-điểm và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle bộ-4-điểm và 9 cài đặt đúng


### B. Edge case

- Một điểm, hai điểm, ba điểm: không có hình chữ nhật ⇒ `0`.
- Bốn điểm thẳng hàng ⇒ `0`.
- Đúng một hình chữ nhật (4 điểm).
- Hình vuông đơn vị: `1`.
- Nhiều hình chữ nhật, nhỏ nhất **không** phải cặp cột kề đầu tiên.
- Các điểm cùng cột/cùng hàng nhiều (lưới `3 × 3`: nhỏ nhất `1`).
- Hình chữ nhật dùng toạ độ `0` và `4·10^4`.
- Điểm thừa gần hình chữ nhật nhưng không đủ góc.

In [17]:
cases = [
    ([[0, 0]], 0), ([[0, 0], [1, 1]], 0), ([[0, 0], [0, 1], [1, 0]], 0), ([[0, 0], [0, 1], [0, 2], [0, 3]], 0),
    ([[0, 0], [0, 2], [3, 0], [3, 2]], 6), ([[0, 0], [0, 1], [1, 0], [1, 1]], 1),
    ([[x, y] for x in range(3) for y in range(3)], 1),
    ([[0, 0], [0, 40000], [40000, 0], [40000, 40000]], 40000 * 40000),
    ([[0, 0], [0, 5], [2, 0], [2, 5], [7, 0], [7, 5], [3, 3]], 10),
    ([[1, 1], [1, 3], [3, 1], [2, 3]], 0),
    ([[0, 0], [0, 4], [4, 0], [4, 4], [1, 1], [1, 3], [3, 1], [3, 3]], 4),
]
for pts, want in cases:
    check(pts, want)
print(len(cases), "edge case: oracle và mọi cài đặt đúng")

11 edge case: oracle và mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [18]:
def bug_degenerate(points):                            # không loại cặp cùng x hoặc cùng y: diện tích 0 được tính
    S = {tuple(p) for p in points}
    best = float("inf")
    for i, (x1, y1) in enumerate(points):
        for x2, y2 in points[i + 1:]:
            if (x1, y2) in S and (x2, y1) in S:         # SAI: thiếu x1 != x2 and y1 != y2
                best = min(best, abs(x2 - x1) * abs(y2 - y1))
    return 0 if best == float("inf") else best


def bug_inf(points):                                   # không có hình chữ nhật: trả inf thay vì 0
    S = {tuple(p) for p in points}
    best = float("inf")
    for i, (x1, y1) in enumerate(points):
        for x2, y2 in points[i + 1:]:
            if x1 != x2 and y1 != y2 and (x1, y2) in S and (x2, y1) in S:
                best = min(best, abs(x2 - x1) * abs(y2 - y1))
    return best


def bug_first_column(points):                          # lưu cột ĐẦU TIÊN thay vì gần nhất
    cols = defaultdict(list)
    for x, y in points:
        cols[x].append(y)
    first, best = {}, float("inf")
    for x in sorted(cols):
        ys = sorted(cols[x])
        for i in range(len(ys)):
            for j in range(i + 1, len(ys)):
                pair = (ys[i], ys[j])
                if pair in first:
                    best = min(best, (x - first[pair]) * (ys[j] - ys[i]))
                else:
                    first[pair] = x                     # SAI: không ghi đè bằng cột gần nhất
    return 0 if best == float("inf") else best


def bug_adjacent_y_only(points):                       # chỉ xét cặp y kề nhau trong cột
    cols = defaultdict(list)
    for x, y in points:
        cols[x].append(y)
    last, best = {}, float("inf")
    for x in sorted(cols):
        ys = sorted(cols[x])
        for i in range(len(ys) - 1):                    # SAI: bỏ các cặp không kề
            pair = (ys[i], ys[i + 1])
            if pair in last:
                best = min(best, (x - last[pair]) * (ys[i + 1] - ys[i]))
            last[pair] = x
    return 0 if best == float("inf") else best


def bug_only_square(points):                           # coi hình chữ nhật chỉ là hình vuông
    S = {tuple(p) for p in points}
    best = float("inf")
    for i, (x1, y1) in enumerate(points):
        for x2, y2 in points[i + 1:]:
            if abs(x1 - x2) == abs(y1 - y2) != 0 and (x1, y2) in S and (x2, y1) in S:
                best = min(best, abs(x2 - x1) * abs(y2 - y1))
    return 0 if best == float("inf") else best


def bug_one_corner_missing(points):                    # chỉ kiểm tra một trong hai đỉnh còn lại
    S = {tuple(p) for p in points}
    best = float("inf")
    for i, (x1, y1) in enumerate(points):
        for x2, y2 in points[i + 1:]:
            if x1 != x2 and y1 != y2 and (x1, y2) in S:      # SAI: thiếu (x2, y1) in S
                best = min(best, abs(x2 - x1) * abs(y2 - y1))
    return 0 if best == float("inf") else best


BUGS = [bug_degenerate, bug_inf, bug_first_column, bug_adjacent_y_only, bug_only_square, bug_one_corner_missing]
rng = random.Random(3)
tests = []
while len(tests) < 800:
    tests.append(random_points(rng, rng.randint(1, 9), 3, 3))
for bug in BUGS:
    bad = next((p for p in tests if bug(p) != oracle(p)), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:22s} sai trên {sorted(map(tuple, bad))}: ra {bug(bad)}, đúng {oracle(bad)}")

bug_degenerate         sai trên [(0, 0), (0, 2), (1, 3), (2, 1), (2, 2), (3, 0), (3, 2)]: ra 0, đúng 6
bug_inf                sai trên [(0, 3), (1, 2), (2, 1), (3, 0)]: ra inf, đúng 0
bug_first_column       sai trên [(0, 1), (0, 2), (0, 3), (1, 2), (2, 1), (2, 3), (3, 0), (3, 1), (3, 3)]: ra 4, đúng 2
bug_adjacent_y_only    sai trên [(0, 1), (0, 2), (1, 0), (2, 1), (2, 3), (3, 0), (3, 1), (3, 2), (3, 3)]: ra 3, đúng 2
bug_only_square        sai trên [(0, 0), (0, 2), (1, 3), (2, 1), (2, 2), (3, 0), (3, 2)]: ra 0, đúng 6
bug_one_corner_missing sai trên [(0, 1), (2, 0), (2, 3), (3, 1), (3, 3)]: ra 2, đúng 0


### D. Vét cạn mọi tập con của lưới nhỏ

Mọi tập con của lưới `3 × 3` (`2⁹ = 512`) và của lưới `3 × 4` (`2¹² = 4 096`), oracle bộ 4 điểm (`C(12,4) = 495` bộ tối đa mỗi tập). Tổng `4 608` tập điểm — bao gồm mọi cấu hình hình chữ nhật chồng lấn trên lưới nhỏ.

In [19]:
t0 = time.perf_counter()
cnt = 0
fast = ("brute", "diagonal", "columns", "colpairs", "Solution")
for W, H in [(3, 3), (3, 4)]:
    cells = [[x, y] for x in range(W) for y in range(H)]
    for mask in range(1, 1 << len(cells)):
        pts = [cells[i] for i in range(len(cells)) if mask >> i & 1]
        want = oracle(pts)
        for name in fast:
            assert IMPLS[name](pts) == want, (name, pts)
        cnt += 1
print(cnt, "tập con (bỏ tập rỗng): 5 cài đặt khớp oracle bộ-4-điểm;", round(time.perf_counter() - t0, 1), "s")
assert cnt == (512 - 1) + (4096 - 1)

4606 tập con (bỏ tập rỗng): 5 cài đặt khớp oracle bộ-4-điểm; 0.2 s


### E. Ngẫu nhiên có seed và tính chất

Điểm ngẫu nhiên trên lưới vừa (so oracle với `n ≤ 14`, so Cách 2 với `n` lớn hơn). Tính chất: (1) **tịnh tiến** hoặc **hoán vị trục** `(x, y) → (y, x)` không đổi đáp án; (2) **thêm điểm** không làm đáp án tăng (nếu đã `> 0`) — chỉ có thể giảm hoặc từ `0` thành dương; (3) đáp án `> 0` ⇒ tồn tại đủ 4 đỉnh (kiểm chứng trực tiếp bằng cách tìm lại hình chữ nhật); (4) nhân toạ độ `x` với `c` nhân đáp án với `c`.

In [20]:
rng = random.Random(20240939)
for _ in range(1200):
    n = rng.randint(1, 14)
    X, Y = rng.choice([(3, 3), (5, 5), (8, 4), (20, 20)])
    pts = random_points(rng, min(n, (X + 1) * (Y + 1)), X, Y)
    want = oracle(pts)
    for name in ("diagonal", "columns", "colpairs", "Solution", "SolutionDiagonal", "SolutionColumns", "SolutionColumnPairs"):
        assert IMPLS[name](pts) == want, (name, pts)
    dx, dy = rng.randint(0, 100), rng.randint(0, 100)
    assert Solution().minAreaRect([[x + dx, y + dy] for x, y in pts]) == want                   # (1) tịnh tiến
    assert Solution().minAreaRect([[y, x] for x, y in pts]) == want                              # (1) hoán vị trục
    c = rng.randint(2, 5)
    assert Solution().minAreaRect([[x * c, y] for x, y in pts]) == want * c                      # (4)
    extra = [rng.randint(0, X), rng.randint(0, Y)]
    if extra not in pts:
        more = Solution().minAreaRect(pts + [extra])
        assert want == 0 or (more != 0 and more <= want)                                         # (2)
    if want > 0:                                                                                  # (3) tìm lại hình chữ nhật có diện tích = want
        S = {tuple(p) for p in pts}
        assert any(x1 != x2 and y1 != y2 and (x1, y2) in S and (x2, y1) in S and abs(x2 - x1) * abs(y2 - y1) == want
                   for (x1, y1) in S for (x2, y2) in S)
for _ in range(100):
    pts = random_points(rng, rng.randint(30, 120), 15, 15)
    assert Solution().minAreaRect(pts) == min_area_columns(pts) == min_area_column_pairs(pts) == min_area_diagonal(pts)
print("1200 tập điểm ngẫu nhiên khớp oracle; 4 tính chất (tịnh tiến/đối xứng trục, thêm điểm, tìm lại hình, tỉ lệ) và 100 tập tới 120 điểm đúng")

1200 tập điểm ngẫu nhiên khớp oracle; 4 tính chất (tịnh tiến/đối xứng trục, thêm điểm, tìm lại hình, tỉ lệ) và 100 tập tới 120 điểm đúng


### F. Giới hạn: `n = 500`

Các cách `O(n²)` khớp nhau trên các dạng dữ liệu cực đoan: lưới đặc (nhiều hình chữ nhật), điểm thưa trên miền `4·10^4`, các điểm thẳng hàng (một cột / một hàng), hai cột.

In [21]:
rng = random.Random(500)
fast = ["diagonal", "columns", "colpairs", "Solution", "SolutionDiagonal", "SolutionColumns", "SolutionColumnPairs"]
datasets = {
    "lưới đặc 23x22 cắt 500": [[x, y] for x in range(23) for y in range(22)][:500],
    "thưa 4·10^4": random_points(rng, 500, 40000, 40000),
    "một cột": [[7, y] for y in range(500)],
    "một hàng": [[x, 7] for x in range(500)],
    "hai cột 250": [[0, y] for y in range(250)] + [[40000, y * 2] for y in range(250)],
    "đường chéo": [[i, i] for i in range(500)],
    "hình chữ nhật lớn + nhiễu": [[0, 0], [0, 40000], [40000, 0], [40000, 40000]] + random_points(rng, 496, 30, 30),
}
for name, pts in datasets.items():
    pts = [list(p) for p in {tuple(p) for p in pts}]
    res = {k: IMPLS[k](pts) for k in fast}
    assert len(set(res.values())) == 1, (name, res)
    print(f"{name:28s}: n = {len(pts):3d}, đáp án {res['diagonal']}")
assert Solution().minAreaRect(datasets["một cột"]) == 0 and Solution().minAreaRect(datasets["đường chéo"]) == 0
assert Solution().minAreaRect([list(p) for p in datasets["lưới đặc 23x22 cắt 500"]]) == 1
assert Solution().minAreaRect([[0, y] for y in range(250)] + [[40000, y * 2] for y in range(250)]) == 40000 * 2
print("công thức đóng khớp")

lưới đặc 23x22 cắt 500      : n = 500, đáp án 1
thưa 4·10^4                 : n = 500, đáp án 0


một cột                     : n = 500, đáp án 0


một hàng                    : n = 500, đáp án 0
hai cột 250                 : n = 500, đáp án 80000
đường chéo                  : n = 500, đáp án 0


hình chữ nhật lớn + nhiễu   : n = 500, đáp án 1


công thức đóng khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1: không loại cặp cùng `x` hoặc cùng `y`

In [22]:
pts = [[0, 0], [0, 1], [1, 5]]
print("thoái hoá:", bug_degenerate(pts), "| đúng:", Solution().minAreaRect(pts))
assert bug_degenerate(pts) == 0 and Solution().minAreaRect(pts) == 0
bad = next(p for p in tests if bug_degenerate(p) != oracle(p))
print("phản ví dụ:", sorted(map(tuple, bad)), "->", bug_degenerate(bad), "| đúng:", oracle(bad))

thoái hoá: 0 | đúng: 0
phản ví dụ: [(0, 0), (0, 2), (1, 3), (2, 1), (2, 2), (3, 0), (3, 2)] -> 0 | đúng: 6


**Vì sao sai:** hai điểm cùng cột cho "hình chữ nhật" diện tích `0` (hai đỉnh trùng nhau sau khi tra tập) lấn át các hình chữ nhật thật. **Đúng:** yêu cầu `x1 != x2 and y1 != y2`.

### Lỗi 2: trả `inf` khi không có hình chữ nhật

In [23]:
print("không có hình:", bug_inf([[0, 0], [1, 1]]), "| đúng:", Solution().minAreaRect([[0, 0], [1, 1]]))
assert bug_inf([[0, 0], [1, 1]]) == float("inf")

không có hình: inf | đúng: 0


**Vì sao sai:** đề yêu cầu `0`. **Đúng:** `0 if best == float("inf") else best`.

### Lỗi 3: lưu cột đầu tiên của cặp `(y1, y2)`

In [24]:
pts = [[1, 1], [1, 3], [3, 1], [3, 3], [4, 1], [4, 3]]
print("cột đầu tiên:", bug_first_column(pts), "| đúng:", Solution().minAreaRect(pts))
assert bug_first_column(pts) != 2 and Solution().minAreaRect(pts) == 2

cột đầu tiên: 4 | đúng: 2


**Vì sao sai:** ví dụ 2: dùng cột `1` làm mốc cho cột `4` cho diện tích `6`, bỏ lỡ cặp kề `(3, 4)` diện tích `2`. **Đúng:** luôn ghi đè `last[pair] = x` bằng cột **gần nhất**.

### Lỗi 4: chỉ xét cặp `y` kề nhau trong cột

In [25]:
bad = next(p for p in tests if bug_adjacent_y_only(p) != oracle(p))
print(sorted(map(tuple, bad)), "->", bug_adjacent_y_only(bad), "| đúng:", oracle(bad))

[(0, 1), (0, 2), (1, 0), (2, 1), (2, 3), (3, 0), (3, 1), (3, 2), (3, 3)] -> 3 | đúng: 2


**Vì sao sai:** hai cao độ tạo hình chữ nhật có thể **không kề** trong một cột (có điểm khác ở giữa). **Đúng:** xét mọi cặp `(y1, y2)` trong cột (Cách 3) hoặc dùng tập (Cách 2).

### Lỗi 5: chỉ nhận hình vuông

In [26]:
pts = [[0, 0], [0, 2], [5, 0], [5, 2]]
print("chỉ hình vuông:", bug_only_square(pts), "| đúng:", Solution().minAreaRect(pts))
assert bug_only_square(pts) == 0 and Solution().minAreaRect(pts) == 10

chỉ hình vuông: 0 | đúng: 10


**Vì sao sai:** đề là hình **chữ nhật**, hai cạnh có thể khác độ dài.

### Lỗi 6 (bonus): chỉ kiểm tra một đỉnh còn lại

In [27]:
pts = [[1, 1], [1, 3], [3, 1], [2, 3]]
print("thiếu một đỉnh:", bug_one_corner_missing(pts), "| đúng:", Solution().minAreaRect(pts))
assert bug_one_corner_missing(pts) != 0 and Solution().minAreaRect(pts) == 0

thiếu một đỉnh: 2 | đúng: 0


**Vì sao sai:** cần **cả hai** đỉnh còn lại `(x1, y2)` và `(x2, y1)`. **Đúng:** kiểm tra cả hai trong tập.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "hình chữ nhật/hình vuông song song trục từ tập điểm" → **hai điểm chéo xác định hình**; tra tập băm. Khi cần tránh `O(n²)` trên mọi cặp: nhóm theo cột và nhớ cặp `(y1, y2)`.

**Mẹo:**

1. `set` các `tuple` để tra `O(1)`.
2. Điều kiện chéo: `x1 != x2` và `y1 != y2`.
3. Dùng cột **gần nhất** để cực tiểu hoá diện tích.
4. Hoán vị trục để số cột ≤ số hàng (giảm số cặp).
5. Trả `0` (không `inf`) khi không có.
6. Oracle: bộ 4 điểm với đúng hai `x`, hai `y`.

**Họ bài cùng pattern:** Minimum Area Rectangle II (963, có thể xoay), Max Points on a Line (149), Valid Square (593), Rectangle Area (223), Count Number of Rectangles (corner rectangles, 750), Largest Rectangle in Histogram (84).

## 15. Biến thể

1. **Diện tích lớn nhất** của hình chữ nhật song song trục.
2. **Đếm số hình chữ nhật** song song trục.
3. **Corner Rectangles (750):** đếm hình chữ nhật trên lưới 0/1 với bốn góc là `1`.
4. **Số hình vuông** song song trục.
5. **Hình chữ nhật có diện tích đúng `A`.**

Cài biến thể 1, 2, 4 và đối chiếu vét cạn:

In [28]:
def max_area_rect(points):
    S = {tuple(p) for p in points}
    best = 0
    for (x1, y1), (x2, y2) in itertools.combinations(sorted(S), 2):
        if x1 < x2 and y1 < y2 and (x1, y2) in S and (x2, y1) in S:
            best = max(best, (x2 - x1) * (y2 - y1))
    return best


def count_rects(points):
    """Mỗi hình chữ nhật có đúng một đường chéo 'x1<x2, y1<y2' (Tây-Nam / Đông-Bắc)."""
    S = {tuple(p) for p in points}
    return sum(1 for (x1, y1), (x2, y2) in itertools.combinations(sorted(S), 2)
               if x1 < x2 and y1 < y2 and (x1, y2) in S and (x2, y1) in S)


def count_squares(points):
    S = {tuple(p) for p in points}
    return sum(1 for (x1, y1), (x2, y2) in itertools.combinations(sorted(S), 2)
               if x1 < x2 and y1 < y2 and x2 - x1 == y2 - y1 and (x1, y2) in S and (x2, y1) in S)


def brute_rects(points):
    res = []
    for quad in itertools.combinations([tuple(p) for p in points], 4):
        xs, ys = sorted({p[0] for p in quad}), sorted({p[1] for p in quad})
        if len(xs) == 2 and len(ys) == 2 and {(x, y) for x in xs for y in ys} == set(quad):
            res.append((xs[1] - xs[0], ys[1] - ys[0]))
    return res


rng = random.Random(15)
for _ in range(400):
    pts = random_points(rng, rng.randint(1, 12), 4, 4)
    rects = brute_rects(pts)
    assert max_area_rect(pts) == max((w * h for w, h in rects), default=0)
    assert count_rects(pts) == len(rects)
    assert count_squares(pts) == sum(1 for w, h in rects if w == h)
print("diện tích lớn nhất, đếm hình chữ nhật, đếm hình vuông: khớp vét cạn trên 400 tập điểm")
assert count_rects([[x, y] for x in range(3) for y in range(3)]) == 9

diện tích lớn nhất, đếm hình chữ nhật, đếm hình vuông: khớp vét cạn trên 400 tập điểm


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi bộ 4 điểm | đúng 2 `x`, 2 `y` | `O(n⁴)` | `O(1)` | oracle |
| 2. Đường chéo + tập băm | tra hai đỉnh còn lại | `O(n²)` | `O(n)` | **đáp án chính** |
| 3. Nhóm theo cột + `last` | cặp `(y1,y2)` → cột gần nhất | `O(Σ k²)` | `O(n²)` xấu | nhiều cột nhỏ |
| 4. Cặp cột + giao `y` | khoảng cách `y` kề trong giao | `O(C²·k)` | `O(n)` | ít cột |

Cả bốn khớp oracle bộ-4-điểm trên 4 608 tập con của lưới nhỏ và 1 200 tập ngẫu nhiên, và khớp nhau ở `n = 500` (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [29]:
print(Solution().minAreaRect([[0, 0], [0, 2], [2, 0], [2, 2]]), Solution().minAreaRect([[0, 0], [1, 1], [2, 2]]), Solution().minAreaRect([[0, 0], [0, 3], [2, 0], [2, 3], [5, 0], [5, 3]]), count_rects([[0, 0], [0, 1], [1, 0], [1, 1], [2, 0], [2, 1]]))

4 0 6 3


### Bài 2 (tính tay)

`points = [[0,0],[0,1],[0,3],[2,0],[2,1],[2,3],[3,1],[3,3]]`. (a) Nhóm theo cột. (b) Các cặp `(y1, y2)` xuất hiện ở ít nhất hai cột. (c) Diện tích nhỏ nhất? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện đường chéo + tập băm:

In [30]:
def diagonal_todo(points):
    S = {tuple(p) for p in points}
    best = float("inf")
    for i, (x1, y1) in enumerate(points):
        for x2, y2 in points[i + 1:]:
            pass        # TODO: nếu x1 != x2 và y1 != y2 và hai đỉnh còn lại thuộc S thì cập nhật best bằng diện tích
    return 0 if best == float("inf") else best


# Khi làm xong, bỏ comment để kiểm tra:
# assert [diagonal_todo(p) for p, _ in examples] == [4, 2]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **nhóm theo cột và `last`**:

In [31]:
class MySolution:
    def minAreaRect(self, points):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().minAreaRect([list(q) for q in p]) == w for p, w in examples)

### Bài 5 (mini challenge): Valid Square (593)

Cho bốn điểm `p1, p2, p3, p4` (có thể **xoay** bất kỳ), kiểm tra có tạo hình vuông hợp lệ không: sáu khoảng cách bình phương gồm `4` cạnh bằng nhau `> 0` và `2` đường chéo bằng nhau gấp đôi cạnh. Viết `valid_square(p1, p2, p3, p4)` và kiểm tra bằng vét cạn các hoán vị.

In [32]:
def valid_square(p1, p2, p3, p4):
    pass            # TODO


# Khi làm xong:
# assert valid_square([0, 0], [1, 1], [1, 0], [0, 1]) and not valid_square([0, 0], [1, 1], [1, 0], [0, 12])

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: hình chữ nhật song song trục cần đủ 4 góc.
- Mức 2: `[[0,0],[0,2],[2,0],[2,2]]` ⇒ `4`; ba điểm chéo ⇒ `0`.
- Mức 3: `4 0 6 3`: ba cột `0, 2, 5` mỗi cột `{0, 3}`: nhỏ nhất `(2−0)·3 = 6`; `count_rects` của lưới `3×2` = `3`.

**Bài 2**

- Mức 1: cột `0: {0,1,3}`, cột `2: {0,1,3}`, cột `3: {1,3}`.
- Mức 2: cặp `(1,3)` có ở cột `0, 2, 3`; `(0,1)` ở cột `0, 2`; `(0,3)` ở cột `0, 2`.
- Mức 3: nhỏ nhất: cột `2` và `3`, cặp `(1,3)`: `(3−2)·2 = 2`.

**Bài 3**

- Mức 1: điều kiện chéo `x1 != x2 and y1 != y2`.
- Mức 2: `(x1, y2) in S and (x2, y1) in S`.
- Mức 3: `best = min(best, abs(x2 - x1) * abs(y2 - y1))`.

**Bài 4**

- Mức 1: `cols = defaultdict(list)`, duyệt `x` tăng dần.
- Mức 2: mọi cặp `(ys[i], ys[j])` trong cột; tra `last`.
- Mức 3: `last[pair] = x` sau khi dùng.

**Bài 5**

- Mức 1: sáu khoảng cách bình phương.
- Mức 2: sắp xếp: `4` nhỏ nhất bằng nhau (cạnh), `2` lớn nhất bằng nhau (đường chéo).
- Mức 3: cạnh `> 0` và `chéo == 2·cạnh`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [33]:
vals = (Solution().minAreaRect([[0, 0], [0, 2], [2, 0], [2, 2]]), Solution().minAreaRect([[0, 0], [1, 1], [2, 2]]), Solution().minAreaRect([[0, 0], [0, 3], [2, 0], [2, 3], [5, 0], [5, 3]]), count_rects([[0, 0], [0, 1], [1, 0], [1, 1], [2, 0], [2, 1]]))
assert vals == (4, 0, 6, 3)
print("Bài 1 OK:", vals)

Bài 1 OK: (4, 0, 6, 3)


### Bài 2

Cặp `(1, 3)` có mặt ở cột `0, 2, 3`: các cặp cột kề `(0, 2)` → `2·2 = 4`, `(2, 3)` → `1·2 = 2`. Nhỏ nhất `2`.

In [34]:
pts = [[0, 0], [0, 1], [0, 3], [2, 0], [2, 1], [2, 3], [3, 1], [3, 3]]
assert Solution().minAreaRect(pts) == oracle(pts) == 2
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [35]:
def diagonal_answer(points):
    S = {tuple(p) for p in points}
    best = float("inf")
    for i, (x1, y1) in enumerate(points):
        for x2, y2 in points[i + 1:]:
            if x1 != x2 and y1 != y2 and (x1, y2) in S and (x2, y1) in S:
                best = min(best, abs(x2 - x1) * abs(y2 - y1))
    return 0 if best == float("inf") else best


assert [diagonal_answer(p) for p, _ in examples] == [4, 2]
for p in tests[:300]:
    assert diagonal_answer(p) == oracle(p)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [36]:
class MySolutionAnswer:
    def minAreaRect(self, points):
        cols = defaultdict(list)
        for x, y in points:
            cols[x].append(y)
        last, best = {}, float("inf")
        for x in sorted(cols):
            ys = sorted(cols[x])
            for i in range(len(ys)):
                for j in range(i + 1, len(ys)):
                    pair = (ys[i], ys[j])
                    if pair in last:
                        best = min(best, (x - last[pair]) * (ys[j] - ys[i]))
                    last[pair] = x
        return 0 if best == float("inf") else best


assert all(MySolutionAnswer().minAreaRect([list(q) for q in p]) == w for p, w in examples)
for p in tests[:300]:
    assert MySolutionAnswer().minAreaRect(p) == oracle(p)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [37]:
def valid_square_answer(p1, p2, p3, p4):
    pts = [p1, p2, p3, p4]
    d = sorted((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2 for a, b in itertools.combinations(pts, 2))
    return d[0] > 0 and d[0] == d[1] == d[2] == d[3] and d[4] == d[5] == 2 * d[0]


def valid_square_brute(p1, p2, p3, p4):
    """Vét cạn: thử mọi thứ tự đỉnh; hợp lệ nếu các góc liên tiếp vuông và các cạnh bằng nhau."""
    for perm in itertools.permutations([p1, p2, p3, p4]):
        a, b, c, d = perm
        vecs = [(b[0] - a[0], b[1] - a[1]), (c[0] - b[0], c[1] - b[1]), (d[0] - c[0], d[1] - c[1]), (a[0] - d[0], a[1] - d[1])]
        lens = [v[0] ** 2 + v[1] ** 2 for v in vecs]
        dots = [vecs[i][0] * vecs[(i + 1) % 4][0] + vecs[i][1] * vecs[(i + 1) % 4][1] for i in range(4)]
        if lens[0] > 0 and len(set(lens)) == 1 and all(x == 0 for x in dots):
            return True
    return False


assert valid_square_answer([0, 0], [1, 1], [1, 0], [0, 1]) and not valid_square_answer([0, 0], [1, 1], [1, 0], [0, 12])
rng = random.Random(5)
for _ in range(500):
    q = [[rng.randint(0, 3), rng.randint(0, 3)] for _ in range(4)]
    assert valid_square_answer(*q) == valid_square_brute(*q)
print("Bài 5 OK: khớp vét cạn các thứ tự đỉnh")

Bài 5 OK: khớp vét cạn các thứ tự đỉnh


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (hình chữ nhật song song trục từ tập điểm → đường chéo + tập băm):**

```python
class Solution:
    def minAreaRect(self, points: List[List[int]]) -> int:
        S = {(x, y) for x, y in points}
        best = float("inf")
        for i in range(len(points)):
            x1, y1 = points[i]
            for j in range(i + 1, len(points)):
                x2, y2 = points[j]
                if x1 != x2 and y1 != y2 and (x1, y2) in S and (x2, y1) in S:
                    best = min(best, abs(x2 - x1) * abs(y2 - y1))
        return 0 if best == float("inf") else best
```

**Checklist trước khi submit:**

- [ ] Điều kiện chéo `x1 != x2 and y1 != y2` (loại cặp thoái hoá)?
- [ ] Kiểm tra **cả hai** đỉnh còn lại `(x1, y2)` và `(x2, y1)` trong tập?
- [ ] Dùng `tuple` trong `set` (không `list`)?
- [ ] Trả `0` (không `inf`) khi không có hình chữ nhật?
- [ ] (Cách theo cột) ghi đè `last[pair]` bằng cột **gần nhất**, xét mọi cặp `y` (không chỉ kề nhau)?
- [ ] Đã thử 2 ví dụ, 3 điểm, các điểm thẳng hàng, lưới `3×3` (`1`), toạ độ `0` và `4·10^4`?
- [ ] Đúng tên `class Solution`, `minAreaRect(self, points)`, không in debug.